# **Data Extraction** : Extract json data from github repo using only python.

In [18]:
# Clone github repo to google colab

!git clone https://github.com/PhonePe/pulse.git

fatal: destination path 'pulse' already exists and is not an empty directory.


In [19]:
# Connect colab with google drive.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
!ls /content/pulse/data

aggregated  map  top


There are three major folders to extract.

In [21]:
# import important libraries.
import os
import json
import pandas as pd
import git

base_path = '/content/pulse/data'

In [22]:
# Tables
agg_transaction = []
agg_insurance = []
agg_user = []

map_transaction = []
map_insurance = []
map_user = []

top_transaction = []
top_insurance = []
top_user = []

# Walk through root, directories, files from all the links.
for root, dirs, files in os.walk(base_path):
    for file in files:

        if not file.endswith(".json"):
            continue

        full_path = os.path.join(root, file)

        try:
            with open(full_path, "r") as f:
                content = json.load(f)
        except:
            continue

        data = content.get("data")
        if not data:
            continue

        parts = full_path.replace("\\", "/").split("/")

        # Meta data extraction like year, quarter, state.
        try:
            year = parts[-2]
            quarter = file.replace(".json", "")

            state = parts[parts.index("state") + 1] if "state" in parts else ' '
        except:
            continue

        # Type of transaction.
        dtype = None
        if "transaction" in parts:
            dtype = "transaction"
        elif "insurance" in parts:
            dtype = "insurance"
        elif "user" in parts:
            dtype = "user"

        # level of the data.
        level = None
        if "aggregated" in parts:
            level = "aggregated"
        elif "map" in parts:
            level = "map"
        elif "top" in parts:
            level = "top"

        # Aggragated (transaction + insurance)
        if level == "aggregated" and data.get("transactionData"):
            for item in data["transactionData"]:
                for pi in item.get("paymentInstruments", []):

                    record = {
                        "state": state,
                        "year": year,
                        "quarter": quarter,
                        "entity_name": item.get("name"),
                        "count": pi.get("count"),
                        "amount": pi.get("amount")
                    }

                    if dtype == "transaction":
                        agg_transaction.append(record)
                    elif dtype == "insurance":
                        agg_insurance.append(record)

        # Aggragated user.
        if level == "aggregated" and data.get("usersByDevice"):
            for item in data["usersByDevice"]:

                record = {
                    "state": state,
                    "year": year,
                    "quarter": quarter,
                    "brand": item.get("brand"),
                    "count": item.get("count"),
                    "percentage": item.get("percentage")
                }

                agg_user.append(record)

        # 🔹 Map (transaction + insurance)
        if level == "map" and data.get("hoverDataList"):
            for item in data["hoverDataList"]:

                metric = item.get("metric")
                metric = metric[0] if isinstance(metric, list) else metric or {}

                record = {
                    "state": state,
                    "year": year,
                    "quarter": quarter,
                    "district": item.get("name"),
                    "count": metric.get("count"),
                    "amount": metric.get("amount")
                }

                if dtype == "transaction":
                    map_transaction.append(record)
                elif dtype == "insurance":
                    map_insurance.append(record)

        # Map user (special case).
        if level == "map" and data.get("hoverData"):
            for district, values in data["hoverData"].items():

                record = {
                    "state": state,
                    "year": year,
                    "quarter": quarter,
                    "district": district,
                    "registered_users": values.get("registeredUsers"),
                    "app_opens": values.get("appOpens")
                }

                map_user.append(record)

        # Top data (all types)
        if level == "top":
          for key in ["states", "districts", "pincodes"]:
              value = data.get(key)

              if isinstance(value, list):
                  for item in value:

                      # User
                      if dtype == "user":
                          record = {
                              "level": key,
                              "entity_name": item.get("name"),
                              "state": state,
                              "year": year,
                              "quarter": quarter,
                              "registered_users": item.get("registeredUsers")
                          }

                          top_user.append(record)

                      # Transaction / Insurance
                      else:
                          metric = item.get("metric")
                          metric = metric[0] if isinstance(metric, list) else metric or {}

                          record = {
                              "level": key,
                              "entity_name": item.get("name") or item.get("entityName"),
                              "state": state,
                              "year": year,
                              "quarter": quarter,
                              "count": metric.get("count"),
                              "amount": metric.get("amount")
                          }

                          if dtype == "transaction":
                              top_transaction.append(record)
                          elif dtype == "insurance":
                              top_insurance.append(record)

In [23]:
agg_transaction = pd.DataFrame(agg_transaction)
agg_insurance = pd.DataFrame(agg_insurance)
agg_user = pd.DataFrame(agg_user)

map_transaction = pd.DataFrame(map_transaction)
map_insurance = pd.DataFrame(map_insurance)
map_user = pd.DataFrame(map_user)

top_transaction = pd.DataFrame(top_transaction)
top_insurance = pd.DataFrame(top_insurance)
top_user = pd.DataFrame(top_user)

In [24]:
agg_transaction.head()

,state,year,quarter,entity_name,count,amount
0,,2023,2,Merchant payments,8529595637,4.277821e+12
1,,2023,2,Peer-to-peer payments,5374546962,1.772278e+13
2,,2023,2,Recharge & bill payments,1190947064,9.047589e+11
3,,2023,2,Financial Services,5195942,6.364034e+09
4,,2023,2,Others,9341245,6.919678e+09


In [25]:
# Check all the datasets.

lst = [agg_transaction, agg_insurance, agg_user, map_transaction, map_insurance, map_user, top_insurance, top_transaction, top_user]

for i in lst:
  print(i.head())

  state  year quarter               entity_name       count        amount
0        2023       2         Merchant payments  8529595637  4.277821e+12
1        2023       2     Peer-to-peer payments  5374546962  1.772278e+13
2        2023       2  Recharge & bill payments  1190947064  9.047589e+11
3        2023       2        Financial Services     5195942  6.364034e+09
4        2023       2                    Others     9341245  6.919678e+09
  state  year quarter entity_name    count        amount
0        2023       2   Insurance   893850  1.347400e+09
1        2023       3   Insurance  1010211  1.445235e+09
2        2023       4   Insurance  1159063  1.855300e+09
3        2023       1   Insurance   923173  1.408801e+09
4        2021       2   Insurance   363989  2.950667e+08
  state  year quarter    brand     count  percentage
0        2021       2   Xiaomi  75948492    0.248801
1        2021       2  Samsung  59144667    0.193753
2        2021       2     Vivo  58646521    0.192121
3 

In [26]:
# save all the datasets.

name = ["agg_transaction", "agg_insurance", "agg_user", "map_transaction", "map_insurance", "map_user", "top_insurance", "top_transaction", "top_user"]

for i in range(len(lst)):
  lst[i].to_csv(f'/content/drive/MyDrive/Phonepe/{name[i]}.csv', index = False)